In [1]:
import pandas as pd
import os
from pathlib import Path
import json

def concatenate_csv_files(folder_path: str) -> pd.DataFrame:
    """
    Read and concatenate all CSV files in the specified folder.
    
    Args:
        folder_path: Path to folder containing CSV files
        
    Returns:
        Concatenated DataFrame of all CSVs
    """
    # Get all CSV files in the folder
    csv_files = list(Path(folder_path).glob('*.csv'))
    
    if not csv_files:
        raise ValueError(f"No CSV files found in {folder_path}")
    
    # Read and concatenate all CSV files
    dfs = []
    for file in csv_files:
        df = pd.read_csv(file)
        dfs.append(df)
    
    # Concatenate all dataframes
    combined_df = pd.concat(dfs, ignore_index=True)
    
    
    return combined_df


def format_questions_as_json(df: pd.DataFrame, folder_path: str) -> dict:
    """
    Transform DataFrame into JSON format for hard questions.
    
    Args:
        df: DataFrame containing question data
        
    Returns:
        Dictionary ready to be saved as JSON
    """
    questions_list = []
    
    for question_id in df['question_triple_id'].unique():
        triple_rows = df[df['question_triple_id'] == question_id]
        
        if len(triple_rows) != 3:  # Skip incomplete triples
            continue
            
        # Get common information
        score = float(triple_rows['consistency_score'].iloc[0])
        hypothesis = triple_rows['hypothesis'].iloc[0]
        
        # Get questions by type
        p_row = triple_rows[triple_rows['question_type'] == 'P'].iloc[0]
        q_given_p_row = triple_rows[triple_rows['question_type'] == 'Q_given_P'].iloc[0]
        p_and_q_row = triple_rows[triple_rows['question_type'] == 'P_and_Q'].iloc[0]
        
        question_dict = {
            "consistency_score": score,
            "hypothesis": hypothesis,
            "questions": {
                "P": {
                    "title": p_row['question_title'],
                    "body": p_row['question_body']
                },
                "Q_given_P": {
                    "title": q_given_p_row['question_title'],
                    "body": q_given_p_row['question_body']
                },
                "P_and_Q": {
                    "title": p_and_q_row['question_title'],
                    "body": p_and_q_row['question_body']
                }
            }
        }
        questions_list.append(question_dict)
    
    # Sort by consistency score in descending order
    questions_list.sort(key=lambda x: x['consistency_score'], reverse=True)
    
    output_dict = {
        "hard_questions": questions_list
    }
    
    # Save to JSON file
    with open(f'{folder_path}/hard_brute_force_questions_2.json', 'w') as f:
        json.dump(output_dict, f, indent=2)
    
    return output_dict


In [2]:
folder_path = "/Users/bprithvi/Desktop/Research Projects/Brachio/adaptive_evals/tasks/cond_100/gpt-4o-mini"  
combined_df = concatenate_csv_files(folder_path)

In [3]:
hard_df = combined_df[combined_df.consistency_score > 0.30].sort_values(by="consistency_score", ascending=False)

In [4]:
formatted_questions = format_questions_as_json(hard_df, folder_path=folder_path)

In [5]:
#GPT-4o brute force generated questions
final_df = combined_df[combined_df.consistency_score > 0.30].sort_values(by="consistency_score", ascending=False)

In [84]:
final_df.to_csv("deepseek-v3_dataset.csv")

In [81]:
format_questions_as_json(final_df, folder_path=folder_path)

{'hard_questions': [{'consistency_score': 0.7637274062416217,
   'hypothesis': "The model assumes a strong correlation between a country's investment in renewable energy and its reduction in greenhouse gas emissions, particularly in the context of climate change mitigation",
   'questions': {'P': {'title': 'Will the European Union implement a carbon border adjustment mechanism by 2028?',
     'body': "This question resolves as YES if the European Union implements a carbon border adjustment mechanism by 2028, as reported by the European Commission on their official website (ec.europa.eu) by December 31, 2028. Resolution source: European Commission's official press releases."},
    'Q_given_P': {'title': 'Given that the European Union implements a carbon border adjustment mechanism, will the global demand for renewable energy increase by more than 12% by 2032?',
     'body': "This question resolves as YES if the global demand for renewable energy increases by more than 12% by 2032, given

In [82]:
hard_df.consistency_score.mean()

0.4716048115907798

In [83]:
final_df.consistency_score.mean()

0.4716048115907798

In [61]:
final_df

,question_triple_id,iteration,hypothesis,topic,reasoning_flaw,question_type,question_title,question_body,avg_forecast,individual_forecasts,consistency_score
72,iter2_h4_q1,2,The model overestimates the correlation betwee...,Biotechnology,NaN,P,Will CRISPR Therapeutics invest more than $2.2...,This question resolves as YES if CRISPR Therap...,0.150,"[0.15, 0.15, 0.15, 0.15, 0.15]",0.929674
73,iter2_h4_q1,2,The model overestimates the correlation betwee...,Biotechnology,NaN,Q_given_P,Given that CRISPR Therapeutics invests more th...,This question resolves as YES if CRISPR Therap...,0.830,"[0.75, 0.95, 0.75, 0.95, 0.75]",0.929674
74,iter2_h4_q1,2,The model overestimates the correlation betwee...,Biotechnology,NaN,P_and_Q,Will CRISPR Therapeutics invest more than $2.2...,This question resolves as YES if CRISPR Therap...,0.650,"[0.65, 0.65, 0.65, 0.65, 0.65]",0.929674
9,iter1_h1_q0,1,The model underestimates the impact of quantum...,Quantum Computing,Underestimation of impact,P,Will IBM invest more than $6 billion in quantu...,This question resolves as YES if IBM's officia...,0.270,"[0.25, 0.15, 0.35, 0.25, 0.35]",0.922361
10,iter1_h1_q0,1,The model underestimates the impact of quantum...,Quantum Computing,Underestimation of impact,Q_given_P,Given that IBM invests more than $6 billion in...,This question resolves as YES if IBM's officia...,0.530,"[0.45, 0.65, 0.65, 0.45, 0.45]",0.922361
...,...,...,...,...,...,...,...,...,...,...,...
25,iter1_h2_q2,1,The model underestimates the impact of quantum...,Quantum Computing,Underestimation of impact,Q_given_P,Given that Microsoft invests more than $7.5 bi...,This question resolves as YES if Microsoft's o...,0.350,"[0.35, 0.35, 0.35, 0.35, 0.35]",0.307118
24,iter1_h2_q2,1,The model underestimates the impact of quantum...,Quantum Computing,Underestimation of impact,P,Will Microsoft invest more than $7.5 billion i...,This question resolves as YES if Microsoft's o...,0.250,"[0.25, 0.25, 0.25, 0.25, 0.25]",0.307118
203,iter3_h4_q0,3,The model overestimates the correlation betwee...,Technology and Customer Satisfaction,NaN,P_and_Q,Will Walmart invest more than $1 billion in di...,This question resolves as YES if Walmart's off...,0.128,"[0.14, 0.11, 0.14, 0.11, 0.14]",0.303526
202,iter3_h4_q0,3,The model overestimates the correlation betwee...,Technology and Customer Satisfaction,NaN,Q_given_P,Given that Walmart invests more than $1 billio...,This question resolves as YES if the American ...,0.350,"[0.35, 0.35, 0.35, 0.35, 0.35]",0.303526


# UMAP analysis

In [6]:
def load_and_combine_model_datasets(base_path: str, model_names: list[str]) -> pd.DataFrame:
    """
    Load CSV files for each model and combine them into a single DataFrame with model labels.
    
    Args:
        base_path: Base directory path containing the CSV files
        model_names: List of model names to process
        
    Returns:
        Combined DataFrame with generator model labels
    """
    all_dfs = []
    
    for model in model_names:
        file_path = os.path.join(base_path, f"{model}_dataset.csv")
        try:
            df = pd.read_csv(file_path, index_col=0)
            # Add generator model name as a column
            df['generator_model'] = model
            all_dfs.append(df)
        except FileNotFoundError:
            print(f"Warning: Could not find dataset for {model} at {file_path}")
    
    # Combine all dataframes
    combined_df = pd.concat(all_dfs, ignore_index=True)
    return combined_df

# Load and combine all datasets
base_path = "/Users/bprithvi/Desktop/Research Projects/Brachio/adaptive_evals/tasks/cond_100"
model_names = ['gpt-4o', 'deepseek-v3', 'llama_8b', 'llama_70b', 'o1_mini', 'sonnet']

combined_df = load_and_combine_model_datasets(base_path, model_names)

# Display basic statistics
print("\nNumber of questions per model:")
print(combined_df.groupby('generator_model').size())

print("\nAverage consistency score per model:")
print(combined_df.groupby('generator_model')['consistency_score'].mean())


Number of questions per model:
generator_model
deepseek-v3    201
gpt-4o          93
llama_70b      201
llama_8b       186
o1_mini        210
sonnet         198
dtype: int64

Average consistency score per model:
generator_model
deepseek-v3    0.471605
gpt-4o         0.387801
llama_70b      0.448420
llama_8b       0.423736
o1_mini        0.454586
sonnet         0.487150
Name: consistency_score, dtype: float64


In [7]:
logical_and_df = combined_df[combined_df.question_type == "P_and_Q"]
logical_and_df.head()

,question_triple_id,iteration,hypothesis,topic,reasoning_flaw,question_type,question_title,question_body,avg_forecast,individual_forecasts,consistency_score,generator_model,generation_reasoning
2,iter3_h4_q1,3,The model assumes a strong correlation between...,Business and Technology,NaN,P_and_Q,Microsoft's research and development expenses ...,Will both Microsoft increase its research and ...,0.65,"[0.65, 0.65, 0.65, 0.65, 0.65]",0.689318,gpt-4o,NaN
5,iter8_h2_q0,8,The model overestimates the correlation betwee...,Business and Environment,NaN,P_and_Q,Joint Success of Microsoft's Renewable Energy ...,Will both Microsoft's investment in renewable ...,0.65,"[0.65, 0.65, 0.65, 0.65, 0.65]",0.517153,gpt-4o,NaN
6,iter3_h2_q0,3,The model assumes a strong correlation between...,Economics and Technology,NaN,P_and_Q,China's investment in artificial intelligence ...,Will both China increase its investment in art...,0.57,"[0.55, 0.55, 0.65, 0.55, 0.55]",0.488011,gpt-4o,NaN
11,iter9_h2_q0,9,The model overestimates the correlation betwee...,Environmental Sustainability,NaN,P_and_Q,Joint Success of Microsoft's Renewable Energy ...,Will both Microsoft's investment in renewable ...,0.63,"[0.65, 0.65, 0.55, 0.65, 0.65]",0.482216,gpt-4o,NaN
12,iter4_h3_q0,4,The model underestimates the impact of climate...,Agriculture and Climate,NaN,P_and_Q,Joint Impact of Temperature Increase and Wheat...,Will both the global average temperature incre...,0.31,"[0.35, 0.35, 0.25, 0.35, 0.25]",0.460263,gpt-4o,NaN


In [8]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import itertools

def compute_model_similarities(logical_and_df: pd.DataFrame) -> dict:
    """
    Compute similarities between different models' questions using TF-IDF and cosine similarity.
    
    Args:
        logical_and_df: DataFrame containing P_and_Q questions with generator_model labels
        
    Returns:
        Dictionary containing average and max similarity between model pairs
    """
    # Group questions by model
    model_texts = {}
    for model in logical_and_df['generator_model'].unique():
        model_questions = logical_and_df[logical_and_df['generator_model'] == model]['question_body'].tolist()
        model_texts[model] = model_questions
    
    # Convert texts to TF-IDF vectors
    vectorizer = TfidfVectorizer(stop_words='english')
    model_vectors = {}
    
    # Create a combined corpus for fitting the vectorizer
    all_texts = [text for texts in model_texts.values() for text in texts]
    vectorizer.fit(all_texts)
    
    # Transform each model's questions
    for model, texts in model_texts.items():
        vectors = vectorizer.transform(texts)
        model_vectors[model] = vectors
    
    # Compute pairwise similarities
    similarities = {}
    for model1, model2 in itertools.combinations(model_vectors.keys(), 2):
        # Compute cosine similarities between all question pairs
        sims = cosine_similarity(model_vectors[model1], model_vectors[model2])
        avg_sim = np.mean(sims)
        max_sim = np.max(sims)
        
        pair_name = f"{model1}_vs_{model2}"
        similarities[pair_name] = {
            'avg_similarity': avg_sim,
            'max_similarity': max_sim
        }
    
    return similarities

# Compute similarities
similarities = compute_model_similarities(logical_and_df)

# Print results sorted by average similarity (lowest first = most different)
print("\nModel pairs sorted by average similarity (lowest = most different):")
sorted_by_avg = sorted(similarities.items(), 
                      key=lambda x: x[1]['avg_similarity'])
print("\nTop 5 most different model pairs by average:")
for pair, scores in sorted_by_avg[:5]:
    print(f"{pair}:")
    print(f"  Average similarity: {scores['avg_similarity']:.3f}")
    print(f"  Maximum similarity: {scores['max_similarity']:.3f}")

# Print results sorted by maximum similarity
print("\nTop 5 most different model pairs by maximum:")
sorted_by_max = sorted(similarities.items(), 
                      key=lambda x: x[1]['max_similarity'])
for pair, scores in sorted_by_max[:5]:
    print(f"{pair}:")
    print(f"  Maximum similarity: {scores['max_similarity']:.3f}")
    print(f"  Average similarity: {scores['avg_similarity']:.3f}")


Model pairs sorted by average similarity (lowest = most different):

Top 5 most different model pairs by average:
gpt-4o_vs_llama_8b:
  Average similarity: 0.029
  Maximum similarity: 0.397
gpt-4o_vs_llama_70b:
  Average similarity: 0.029
  Maximum similarity: 0.538
gpt-4o_vs_deepseek-v3:
  Average similarity: 0.031
  Maximum similarity: 0.734
gpt-4o_vs_sonnet:
  Average similarity: 0.036
  Maximum similarity: 0.622
gpt-4o_vs_o1_mini:
  Average similarity: 0.038
  Maximum similarity: 0.365

Top 5 most different model pairs by maximum:
llama_8b_vs_sonnet:
  Maximum similarity: 0.271
  Average similarity: 0.053
llama_8b_vs_o1_mini:
  Maximum similarity: 0.315
  Average similarity: 0.051
llama_70b_vs_sonnet:
  Maximum similarity: 0.348
  Average similarity: 0.072
gpt-4o_vs_o1_mini:
  Maximum similarity: 0.365
  Average similarity: 0.038
gpt-4o_vs_llama_8b:
  Maximum similarity: 0.397
  Average similarity: 0.029


In [9]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import HDBSCAN
from sklearn.manifold import TSNE
import plotly.express as px
import plotly.graph_objects as go
from collections import Counter

def analyze_model_topics(logical_and_df: pd.DataFrame, model_names: list[str], n_clusters: int = 10):
    """
    Analyze and visualize topic distribution between two models using t-SNE and clustering.
    Interactive visualization with Plotly.
    """
    # Filter for specified models
    model_df = logical_and_df[logical_and_df['generator_model'].isin(model_names)]
    
    # Prepare text data
    texts = model_df['question_body'].tolist()
    
    # TF-IDF vectorization
    vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
    tfidf_matrix = vectorizer.fit_transform(texts)
    
    # t-SNE dimensionality reduction
    tsne = TSNE(n_components=2, 
                perplexity=30, 
                random_state=42)
    
    tsne_embeddings = tsne.fit_transform(tfidf_matrix.toarray())
    
    # Create visualization DataFrame
    viz_df = pd.DataFrame({
        'TSNE1': tsne_embeddings[:, 0],
        'TSNE2': tsne_embeddings[:, 1],
        'Model': model_df['generator_model'].values,
        'Question': texts,  # Full question text for hover
        'Question_Preview': [t[:100] + '...' for t in texts],  # Preview for hover
        'Topic': model_df['topic'].values  # Use existing topic column
    })
    
    # Create interactive plot
    fig = px.scatter(
        viz_df,
        x='TSNE1',
        y='TSNE2',
        color='Model',
        symbol='Model',
        hover_data={
            'TSNE1': False,  # Hide these in hover tooltip
            'TSNE2': False,
            'Model': True,
            'Topic': True,
            'Question_Preview': True
        },
        title='Question Distribution by Model and Topic Cluster',
        labels={'Question_Preview': 'Question'},
        height=800
    )
    
    # Add topic labels
    # Calculate mean position for each topic
    for topic in viz_df['Topic'].unique():
        topic_points = viz_df[viz_df['Topic'] == topic]
        x_mean = topic_points['TSNE1'].mean()
        y_mean = topic_points['TSNE2'].mean()
        
        # Add text annotation
        fig.add_annotation(
            x=x_mean,
            y=y_mean,
            text=topic,
            showarrow=True,
            arrowhead=1,
            arrowsize=1,
            arrowwidth=2,
            arrowcolor="#636363",
            font=dict(size=12, color="#636363"),
            bgcolor="white",
            bordercolor="#c7c7c7",
            borderwidth=1,
            borderpad=4,
            opacity=0.8
        )
    
    # Update layout for better visualization
    fig.update_traces(
        marker=dict(size=10),
        selector=dict(mode='markers')
    )
    
    fig.update_layout(
        legend=dict(
            yanchor="top",
            y=0.99,
            xanchor="left",
            x=0.01
        ),
        plot_bgcolor='white', 
    )
    
    # Show plot
    fig.show()
    
    # Print topic distribution by model
    print("\nTopic distribution by model:")
    for model in model_names:
        model_data = viz_df[viz_df['Model'] == model]
        topic_dist = model_data['Topic'].value_counts()
        print(f"\n{model} topic distribution:")
        print(topic_dist)
    
    return viz_df

# Run the analysis
model_names = ['o1_mini', 'deepseek-v3']
viz_df = analyze_model_topics(logical_and_df, model_names)


Topic distribution by model:

o1_mini topic distribution:
Topic
AI and Computing                         11
Energy and Environment                   10
Telecommunications                        6
Biotechnology and Medicine                4
Biotechnology                             3
Genetic Engineering                       3
Energy                                    3
Telecommunications and Technology         2
Telecommunications and IoT                2
Finance and Technology                    2
Transportation and Logistics              2
Transportation and Energy                 2
Agriculture and Environment               2
Transportation                            1
Technology and Society                    1
Geopolitics and Environment               1
Regulation and Pharmaceuticals            1
Renewable Energy and Economics            1
Biotechnology and Healthcare              1
Finance                                   1
Robotics                                  1
Cybersecuri

In [14]:
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.manifold import TSNE

def analyze_model_topics(logical_and_df: pd.DataFrame, model_names: list[str]):
    """
    Analyze and visualize topic distribution between models using t-SNE.
    Interactive visualization with Plotly with external annotations.
    """
    # Filter for specified models
    model_df = logical_and_df[logical_and_df['generator_model'].isin(model_names)]
    
    # Prepare text data
    texts = model_df['question_body'].tolist()
    
    # TF-IDF vectorization
    vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
    tfidf_matrix = vectorizer.fit_transform(texts)
    
    # t-SNE dimensionality reduction
    tsne = TSNE(n_components=2, 
                perplexity=30, 
                random_state=42)
    
    tsne_embeddings = tsne.fit_transform(tfidf_matrix.toarray())
    
    # Create visualization DataFrame
    viz_df = pd.DataFrame({
        'TSNE1': tsne_embeddings[:, 0],
        'TSNE2': tsne_embeddings[:, 1],
        'Model': model_df['generator_model'].values,
        'Question': texts,
        'Question_Preview': [t[:100] + '...' for t in texts],
        'Topic': model_df['topic'].values
    })

    # Create figure with expanded dimensions
    fig = go.Figure()

    # Add traces for each model
    colors = {'o1_mini': '#00cc96', 'deepseek-v3': '#ff7f7f'}
    symbols = {'o1_mini': 'diamond', 'deepseek-v3': 'circle'}
    
    for model in model_names:
        mask = viz_df['Model'] == model
        fig.add_trace(go.Scatter(
            x=viz_df[mask]['TSNE1'],
            y=viz_df[mask]['TSNE2'],
            mode='markers',
            name=model,
            marker=dict(
                color=colors[model],
                size=8,
                symbol=symbols[model]
            ),
            customdata=np.stack((
                viz_df[mask]['Model'],
                viz_df[mask]['Topic'],
                viz_df[mask]['Question_Preview']
            ), axis=1),
            hovertemplate="<br>".join([
                "Model: %{customdata[0]}",
                "Topic: %{customdata[1]}",
                "Question: %{customdata[2]}",
                "<extra></extra>"
            ])
        ))

    # List of topics to label
    topics_to_label = [
        "Artificial General Intelligence",
        "Sports",
        "Environmental Science",
        "Space Exploration",
        "Biotechnology",
        "Social Media and Politics",
        "Transportation and Logistics"
    ]

    # Create annotations for topic labels and detailed analysis
    annotations = []
    
    # Add basic topic labels
    for topic in topics_to_label:
        topic_points = viz_df[viz_df['Topic'] == topic]
        if len(topic_points) > 0:
            x_mean = topic_points['TSNE1'].mean()
            y_mean = topic_points['TSNE2'].mean()
            
            # Calculate arrow direction - point to the center of the cluster
            arrow_direction = 40  # Base arrow length
            
            annotations.append(dict(
                x=x_mean,
                y=y_mean,
                text=f"<b>{topic}</b>",
                showarrow=True,
                arrowhead=2,
                arrowsize=1,
                arrowwidth=2,
                arrowcolor="#666",
                ax=arrow_direction,
                ay=arrow_direction,
                bgcolor="white",
                bordercolor="#666",
                borderwidth=2,
                borderpad=4,
                font=dict(size=12)
            ))

    # Add detailed analysis boxes outside the plot
    # detailed_annotations = [
    #     # Sports Analysis Box
    #     dict(
    #         x=max(viz_df['TSNE1']) + 50,  # Position far right
    #         y=0,
    #         text=(
    #             "<b>Sports Analysis</b><br><br>"
    #             "Question Generation Strategy:<br>"
    #             "• Inconsistent conditional probability estimates<br>"
    #             "• Poor correlation between individual/team performance<br><br>"
    #             "Example Questions:<br>"
    #             "• 'Will Duplantis break record AND Sweden win 2+ medals?'<br>"
    #             "• 'Will Kipchoge win London AND Kenya win 7+ golds?'"
    #         ),
    #         showarrow=False,
    #         bgcolor="white",
    #         bordercolor="#666",
    #         borderwidth=2,
    #         borderpad=10,
    #         font=dict(size=11),
    #         align="left"
    #     ),
    #     # Transportation Analysis Box
    #     dict(
    #         x=min(viz_df['TSNE1']) - 50,  # Position far left
    #         y=0,
    #         text=(
    #             "<b>Transportation & Logistics Analysis</b><br><br>"
    #             "Reasoning Patterns:<br>"
    #             "• Inconsistent probabilistic reasoning<br>"
    #             "• ECB policy impact assessment<br>"
    #             "• Market reaction analysis<br><br>"
    #             "Example Questions:<br>"
    #             "• 'Will ECB maintain rates AND euro depreciate?'<br>"
    #             "• 'Will charging stations increase AND autonomous driving grow?'"
    #         ),
    #         showarrow=False,
    #         bgcolor="white",
    #         bordercolor="#666",
    #         borderwidth=2,
    #         borderpad=10,
    #         font=dict(size=11),
    #         align="left"
    #     )
    # ]
    
    # annotations.extend(detailed_annotations)

    # Update layout with expanded dimensions and margins
    fig.update_layout(
        template='plotly_white',
        width=1400,  # Increased width to accommodate external boxes
        height=800,
        margin=dict(l=200, r=200, t=100, b=100),  # Increased margins
        
        title=dict(
            text='Question Distribution by Target Model and Topic Cluster',
            x=0.5,
            y=0.95,
            xanchor='center',
            yanchor='top',
            font=dict(size=16)
        ),
        
        showlegend=True,
        legend=dict(
            title='Model',
            yanchor="top",
            y=0.99,
            xanchor="right",
            x=0.99,
            bgcolor='rgba(255, 255, 255, 0.8)'
        ),
        
        # Clean axes
        xaxis=dict(
            showgrid=True,
            gridwidth=1,
            gridcolor='#f0f0f0',
            zeroline=True,
            zerolinewidth=1,
            zerolinecolor='#e0e0e0',
            title='TSNE1'
        ),
        yaxis=dict(
            showgrid=True,
            gridwidth=1,
            gridcolor='#f0f0f0',
            zeroline=True,
            zerolinewidth=1,
            zerolinecolor='#e0e0e0',
            title='TSNE2'
        ),
        
        # Add all annotations
        annotations=annotations
    )

    return fig, viz_df

# Run the analysis
model_names = ['o1_mini', 'deepseek-v3']
fig, viz_df = analyze_model_topics(logical_and_df, model_names)

fig.show()